# Установка необходимых библиотек

In [1]:
#Установка необходимых библиотек
!pip install sqlalchemy

In [2]:
!pip install psycopg2

In [13]:
#Импорт необходимых библиотек для работы
#Для анализа данных
import pandas as pd

#Для визуализации
import matplotlib.pyplot as plt
import seaborn as sns

#Для подключения к БД
import os 
from sqlalchemy import create_engine
from dotenv import load_dotenv

In [14]:
#Получаю доступ к БД, используя данные из файла .env
db_config = {
    'user': os.getenv('DB_USER'),
    'pwd': os.getenv('DB_PASSWORD'),
    'host': os.getenv('DB_HOST'),
    'port': os.getenv('DB_PORT'),
    'db': os.getenv('DB_NAME')
}

In [15]:
#Создаю данные для подключения
connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

In [16]:
#Проверяю результат 
db_config

{'user': None, 'pwd': None, 'host': None, 'port': None, 'db': None}

In [7]:
engine = create_engine(connection_string)

ValueError: invalid literal for int() with base 10: 'None'

-- Настройка параметра synchronize_seqscans важна для проверки
WITH set_config_precode AS (
  SELECT set_config('synchronize_seqscans', 'off', true)
)
-- Напишите ваш запрос ниже
SELECT p.user_id,
      p.device_type_canonical,
      p.order_id,
      p.created_dt_msk AS order_dt,
      p.created_ts_msk AS order_ts,
      p.currency_code,
      p.revenue,
      p.tickets_count,
      EXTRACT (DAY FROM (p.created_dt_msk - LAG(p.created_dt_msk) OVER (PARTITION BY p.user_id ORDER BY p.created_dt_msk)))::INTEGER AS days_since_prev,
      p.event_id,
      e.event_name_code AS event_name,
      e.event_type_main,
      p.service_name,
      r.region_name,
      c.city_name
FROM purchases p
JOIN events e USING(event_id)
JOIN city c USING(city_id)
JOIN regions r USING(region_id)
WHERE p.device_type_canonical IN ('mobile', 'desktop') AND e.event_type_main NOT IN ('фильм')
ORDER BY p.user_id;
